# alibz pipeline walkthrough — every stage, visualized

This notebook replays `alibz.pipeline.analyze_spectrum` **one stage at a time** on a
representative spectrum, capturing the intermediate state after every step so you can see
exactly what each stage consumes, what it changes, and what it hands downstream. It
mirrors `alibz/pipeline.py` stage-for-stage (same calls, same kwargs), so it doubles as
executable documentation of the production chain.

```
 load CSV -> PeakyFinder.fit_spectrum          1  blind multi-Voigt fit (+ background)
          -> estimate_wavelength_shift_segments 2  per-detector-segment shift (db frame)
          -> refine_fit [3a, data-only]        3a blend splits / single-merges from model
                                                  evidence; asymmetric family DEFERRED
          -> PeakyIndexerV3.run  [pass 1]       4  whole-pattern solve (provisional posterior)
          -> refine_fit [3b, physics]          3b asymmetric merges adjudicated against
                                                  the retained candidate species
          -> seed_minor_lines                   5  Boltzmann-prior minor lines
                                                   (merge zones excluded)
          -> recover_residual_lines             6  element-agnostic residual recovery
          -> profiles + deblend_shoulders       7  shape QC -> shoulder deblends
          -> PeakyIndexerV3.run  [pass 2]       8  confirm elements (warm start)
          -> iterative deepening               9  for ions quantified from intense peaks:
             (seed + guarded recover +            seed weak lines + recover faint residuals
              solve_at, 3->2 sigma)               near their db lines at falling bars;
                                                   fixed-(T,ne) re-solve each round (no basin
                                                   drift), basin-guarded
          -> profiles + recover_sa_areas       10  SA growth-curve recovery + the merges'
                                                   pre-measured emission ratios
          -> analyze_detections                11  detection report + confounders
```

**Interactive use** — after running all cells, call from any cell:

- `explore(center_nm, span_nm)` — data + every captured stage model + final components
  + database-line rug + windowed peak table;
- `overlay(center, span, stages=('blind', 'final'))` — compare any subset of stage models;
- `components('final', center, span)` — individual Voigt components of one stage;
- `diff_peaks('refined', 'seeded')` — peaks added / removed / moved between stages;
- the **widget cell at the bottom** gives sliders for all of this if `ipywidgets` is installed.

Stage captures live in `STAGES` (fit dicts) and `RESULTS` (indexer `FitResult`s).


In [1]:
import glob, os, time
import numpy as np
import matplotlib.pyplot as plt

from alibz import (PeakyFinder, refine_fit, seed_minor_lines,
                   analyze_peak_profiles, deblend_shoulders, recover_sa_areas,
                   element_shape_quality, profile_summary,
                   analyze_detections, estimate_peak_uncertainties,
                   plot_spectrum_overview, element_color, element_sort_key)
from alibz.minor_lines import match_and_scale, recover_residual_lines
from alibz.peaky_indexer_v3 import PeakyIndexerV3
from alibz.pipeline import (CONFIDENT_MIN_REFS, DEEPEN_BARS,
                            ESTABLISHED_MIN_FRACTION, SUPPORT_GA_FLOOR,
                            SUPPORT_TOL_NM, composition_collapsed,
                            load_spectrum_csv, sample_name, _halpha_ne)
from alibz.utils.database import Database
from alibz.utils.voigt import multi_voigt, voigt_width
from alibz.utils.wavelength import (estimate_wavelength_shift,
                                    estimate_wavelength_shift_segments,
                                    shift_at)

REPO = (os.path.dirname(os.getcwd())
        if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
DB_PATH = os.path.join(REPO, 'db')

# representative spectrum: MW2-112 (the pipeline's canonical validation sample);
# falls back to the in-repo REE sample when the Drive share is not mounted
MW2 = ('/Users/mwhittaker/Library/CloudStorage/GoogleDrive-mwhittaker@lbl.gov/'
       'My Drive/Postdocs/Xuan Cao/Data/LIBS/MW2-112/raw')
_cand = sorted(glob.glob(os.path.join(MW2, '*.csv')))
SPECTRUM_FILE = _cand[0] if _cand else os.path.join(
    REPO, 'data', 'remote_samples', 'REE_01.csv')

N_CALLS = 40                    # indexer optimizer budget (pipeline default)
SEGMENT_EDGES = (365.0, 620.0)  # SciAps 3-segment detector

x, y = load_spectrum_csv(SPECTRUM_FILE)
db = Database(DB_PATH)
print(f"sample: {sample_name(SPECTRUM_FILE)}")
print(f"{x.size} px,  {x[0]:.1f}-{x[-1]:.1f} nm,  max {y.max():.0f} counts")

sample: 1000-Test #1000 10 38, Mar 25 2024.03.25.10.38
23431 px,  180.0-961.0 nm,  max 8071 counts


## Toolkit

Capture registry + the interactive helpers used throughout. Everything takes wavelength
windows so any region can be inspected at any pipeline stage after the run.

In [2]:
STAGES = {}    # label -> fit dict, in pipeline order
RESULTS = {}   # label -> indexer FitResult
COMPOSITIONS = []  # (label, element_fractions) in pipeline order, for the evolution chart


def keep(label, fit):
    STAGES[label] = fit
    return fit


def bg_of(fit):
    return np.asarray(fit.get('background', np.zeros_like(y)), dtype=float)


def model_of(fit, xs=None):
    xs = x if xs is None else xs
    pk = np.atleast_2d(np.asarray(fit['sorted_parameter_array'], dtype=float))
    if pk.size == 0:
        return np.zeros_like(xs)
    return multi_voigt(xs, np.ravel(pk[:, :4]))


_noise_cache = {}
def noise_of(fit):
    key = id(fit)
    if key not in _noise_cache:
        _noise_cache[key] = PeakyFinder._noise_scale_local(y - bg_of(fit))
    return _noise_cache[key]


def overlay(center=None, span=3.0, stages=None, log=False):
    '''Data + per-stage model curves in a window; residual/sigma of the LAST stage listed.'''
    labels = [s for s in (stages or STAGES) if s in STAGES]
    if not labels:
        print('no captured stages yet'); return
    lo, hi = ((x[0], x[-1]) if center is None
              else (center - span / 2.0, center + span / 2.0))
    m = (x >= lo) & (x <= hi)
    ref = STAGES[labels[-1]]
    fig, (a0, a1) = plt.subplots(2, 1, sharex=True, figsize=(11, 5.6),
                                 gridspec_kw={'height_ratios': [3, 1]})
    a0.plot(x[m], (y - bg_of(ref))[m], color='0.15', lw=0.9, label='data - bg')
    for i, lab in enumerate(labels):
        a0.plot(x[m], model_of(STAGES[lab])[m], lw=1.1, alpha=0.9,
                label=lab, color=plt.cm.tab10(i % 10))
    pk = np.atleast_2d(STAGES[labels[-1]]['sorted_parameter_array'])
    if pk.size and (hi - lo) < 25:
        inw = pk[(pk[:, 1] >= lo) & (pk[:, 1] <= hi)]
        for row in inw:
            a0.axvline(row[1], color='0.75', lw=0.5, zorder=0)
    resid = (y - bg_of(ref) - model_of(ref)) / np.maximum(noise_of(ref), 1e-12)
    a1.plot(x[m], resid[m], 'k-', lw=0.6)
    a1.axhspan(-2, 2, color='g', alpha=0.12)
    a1.axhline(0, color='0.5', lw=0.6)
    a0.set_ylabel('counts'); a1.set_ylabel(r'resid [$\sigma$]')
    a1.set_xlabel('wavelength [nm]')
    if log:
        a0.set_yscale('log')
    a0.legend(fontsize=8, ncol=min(4, len(labels) + 1))
    a0.set_title(f"{labels[-1]}: residual panel" if center is None
                 else f"{lo:.2f}-{hi:.2f} nm")
    fig.tight_layout()
    return fig


def components(label, center, span=1.5, ax=None):
    '''Individual Voigt components of one captured stage inside a window.'''
    fit = STAGES[label]
    lo, hi = center - span / 2.0, center + span / 2.0
    m = (x >= lo) & (x <= hi)
    pk = np.atleast_2d(fit['sorted_parameter_array'])
    inw = pk[(pk[:, 1] >= lo - span) & (pk[:, 1] <= hi + span)]
    show = ax is None
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(x[m], (y - bg_of(fit))[m], color='0.15', lw=1.0, label='data - bg')
    ax.plot(x[m], model_of(fit)[m], 'r-', lw=1.0, alpha=0.85, label=f'{label} model')
    for i, row in enumerate(inw):
        comp = multi_voigt(x[m], row[:4])
        ax.fill_between(x[m], comp, alpha=0.25, color=plt.cm.tab20(i % 20))
    for row in inw:
        if lo <= row[1] <= hi:
            ax.annotate(f"{row[1]:.2f}", (row[1], float(model_of(fit, np.array([row[1]]))[0])),
                        fontsize=7, rotation=90, ha='right', va='bottom', color='0.35')
    ax.set_xlim(lo, hi)
    ax.set_xlabel('wavelength [nm]'); ax.set_ylabel('counts')
    ax.legend(fontsize=8)
    ax.set_title(f"{label}: {inw.shape[0]} components near {center:.2f} nm", fontsize=10)
    if show:
        plt.gcf().tight_layout()


def line_rug(ax, lo, hi, elements, min_gA=5e6, kT=0.9):
    '''Database lines (air wavelengths, shifted into the instrument frame) as labeled ticks.'''
    ymax = ax.get_ylim()[1]
    for el in elements:
        if el in db.no_lines:
            continue
        arr = db.lines(el)
        if arr.size == 0:
            continue
        ion, wl, gA, Ek = (arr[:, 0].astype(float), arr[:, 1].astype(float),
                           arr[:, 3].astype(float), arr[:, 5].astype(float))
        wl_i = wl + shift_at(SHIFT, wl)   # db frame -> instrument frame
        keepm = (wl_i >= lo) & (wl_i <= hi) & (gA >= min_gA) & (ion <= 2)
        if not np.any(keepm):
            continue
        s = gA[keepm] * np.exp(-Ek[keepm] / kT)
        order = np.argsort(s)[::-1][:8]      # strongest few per element
        for w0, i0 in zip(wl_i[keepm][order], ion[keepm][order]):
            ax.axvline(w0, ymax=0.06, color=element_color(el), lw=1.6)
            ax.annotate(f"{el} {'I' * int(i0)}", (w0, ymax * 0.995), fontsize=6,
                        rotation=90, ha='center', va='top', color=element_color(el))


def diff_peaks(label_a, label_b, tol=0.02):
    '''Peaks added / removed / moved between two captured stages (matched by center).'''
    A = np.atleast_2d(STAGES[label_a]['sorted_parameter_array'])
    B = np.atleast_2d(STAGES[label_b]['sorted_parameter_array'])
    used = np.zeros(B.shape[0], dtype=bool)
    removed, changed = [], []
    for row in A:
        d = np.abs(B[:, 1] - row[1])
        d[used] = np.inf
        j = int(np.argmin(d)) if d.size else -1
        if j < 0 or d[j] > tol:
            removed.append(row)
            continue
        used[j] = True
        if abs(B[j, 0] - row[0]) > 0.02 * max(abs(row[0]), 1e-9):
            changed.append((row, B[j]))
    added = B[~used]
    print(f"{label_a} -> {label_b}:  {A.shape[0]} -> {B.shape[0]} peaks  "
          f"(+{added.shape[0]} added, -{len(removed)} removed, "
          f"~{len(changed)} area-changed >2%)")
    return added, removed, changed


def comp_chart(entries, title='composition', min_frac=2e-3):
    '''Grouped bars of element fractions for a list of (label, fractions-dict).'''
    els = sorted({el for _, fr in entries for el, f in fr.items() if f >= min_frac},
                 key=element_sort_key)
    nw = len(entries)
    xp = np.arange(len(els))
    fig, ax = plt.subplots(figsize=(1.8 + 0.62 * len(els) * max(nw / 2, 1), 3.8))
    for i, (lab, fr) in enumerate(entries):
        vals = [fr.get(el, 0.0) for el in els]
        ax.bar(xp + (i - (nw - 1) / 2) * 0.8 / nw, vals, width=0.8 / nw,
               label=lab, edgecolor='0.3', linewidth=0.4,
               color=[element_color(el) for el in els],
               alpha=1.0 - 0.55 * (nw - 1 - i) / max(nw - 1, 1))
    ax.set_xticks(xp); ax.set_xticklabels(els)
    ax.set_yscale('log'); ax.set_ylabel('atom fraction')
    ax.set_ylim(bottom=min_frac / 3)
    ax.set_title(title + (f"  ({', '.join(l for l, _ in entries)}; "
                          "lighter = earlier)" if nw > 1 else ''), fontsize=10)
    fig.tight_layout()
    return fig


def explore(center, span=2.0, stages=None, elements=None, min_gA=5e6):
    '''One-stop window inspector: stage overlays + final components + db-line rug + peak table.'''
    labels = [s for s in (stages or STAGES) if s in STAGES]
    fig = overlay(center, span, stages=labels)
    ax0 = fig.axes[0]
    if elements is None:
        latest = COMPOSITIONS[-1][1] if COMPOSITIONS else {}
        elements = [el for el, f in latest.items() if f >= 5e-3]
    line_rug(ax0, center - span / 2, center + span / 2, elements, min_gA=min_gA)
    components(labels[-1], center, span)
    fit = STAGES[labels[-1]]
    pk = np.atleast_2d(fit['sorted_parameter_array'])
    inw = pk[(pk[:, 1] >= center - span / 2) & (pk[:, 1] <= center + span / 2)]
    print(f"{labels[-1]}: peaks in [{center - span/2:.2f}, {center + span/2:.2f}] nm")
    print(f"{'center':>9} {'area':>10} {'sigma':>7} {'gamma':>7} {'FWHM_pm':>8}")
    for row in inw[np.argsort(-np.abs(inw[:, 0]))][:20]:
        print(f"{row[1]:9.3f} {row[0]:10.1f} {row[2]:7.4f} {row[3]:7.4f} "
          f"{1000 * voigt_width(max(row[2], 1e-6), max(row[3], 1e-6)):8.1f}")


print('toolkit ready')

toolkit ready


## Stage 1 — blind fit (`PeakyFinder.fit_spectrum`)

**Consumes** the raw spectrum. **Produces** the background estimate
and the blind multi-Voigt peak table every later stage edits — no atomic physics is
involved yet; peaks are found from the data alone (Fourier detection → per-peak fits →
shoulder pass → global refinement).

What to look for: the background hugging the baseline without swallowing broad line
wings; the residual panel mostly inside the green ±2σ band; per-segment width floors
(the three detector segments have different resolution — the narrowest peaks per segment
estimate it).

In [3]:
t0 = time.time()
finder = PeakyFinder.__new__(PeakyFinder)   # fit_spectrum needs no data directory
blind = keep('blind', finder.fit_spectrum(x, y, subtract_background=True,
                                          plot=False, n_sigma=0))
pk = blind['sorted_parameter_array']
print(f"{pk.shape[0]} peaks in {time.time() - t0:.0f}s")

fig, axs = plot_spectrum_overview(x, y, blind)
axs[0].set_title(f"Stage 1 blind fit — {sample_name(SPECTRUM_FILE)}", fontsize=10)

# per-segment stats: pitch, narrowest FWHM (~instrument floor), peak count
edges = (x[0],) + SEGMENT_EDGES + (x[-1],)
print(f"{'segment':>16} {'pixels':>7} {'pitch_pm':>9} {'peaks':>6} {'min_FWHM_pm':>12}")
for i in range(3):
    m = (x >= edges[i]) & (x < edges[i + 1])
    pm = (pk[:, 1] >= edges[i]) & (pk[:, 1] < edges[i + 1])
    w = [1000 * voigt_width(max(s, 1e-6), max(g, 1e-6))
         for s, g in pk[pm][:, 2:4]] or [float('nan')]
    print(f"{edges[i]:7.0f}-{edges[i+1]:4.0f} nm {m.sum():7d} "
          f"{1000 * np.median(np.diff(x[m])):9.2f} {pm.sum():6d} "
          f"{np.nanmin(w):12.1f}")

303 peaks in 1s
         segment  pixels  pitch_pm  peaks  min_FWHM_pm
    180- 365 nm    5550     33.33    131         78.1
    365- 620 nm    7650     33.33    106        108.6
    620- 961 nm   10230     33.33     66        154.9


## Stage 2 — wavelength shift, two steps

**Consumes** the peak table + the line database. The shift is estimated twice:

1. a pooled **global** median from the *blind* table (`estimate_wavelength_shift`) —
   robust enough for stage 3a's coarse db-evidence windows;
2. **per-detector-segment** shifts from the *refined* table after stage 3a
   (`estimate_wavelength_shift_segments`) — blind centers of split/merged bright lines
   sit up to ~150 pm off, so only refined medians are clean enough to judge. A segment
   keeps its own median only when it deviates from the global by more than twice the
   median's standard error; otherwise it falls back to the global (measured on MW2-112:
   per-segment medians differ by 10–35 pm but with 26–71 pm MADs, so the gate correctly
   refuses — the machinery is armed for sessions where a segment genuinely drifts).

All database matching downstream happens in the *db frame*: peak centers minus their
segment's shift. The histogram shows bright-peak offsets to the nearest strong database
line before and after removing the global shift.

In [4]:
SHIFT0, N_ANCHOR = estimate_wavelength_shift(pk, db)
print(f"global shift = {1000 * SHIFT0:+.1f} pm from {N_ANCHOR} anchor lines")

# illustration: offsets of the brightest peaks to their nearest strong db line
wl_all = []
for el in db.elements:
    if el in db.no_lines:
        continue
    arr = db.lines(el)
    if arr.size == 0:
        continue
    keepm = (arr[:, 0].astype(float) <= 2) & (arr[:, 3].astype(float) >= 1e7)
    wl_all.append(arr[keepm, 1].astype(float))
wl_all = np.sort(np.concatenate(wl_all))
bright = pk[np.argsort(-pk[:, 0])[:60], 1]
near = wl_all[np.clip(np.searchsorted(wl_all, bright), 1, wl_all.size - 1)]
near_lo = wl_all[np.clip(np.searchsorted(wl_all, bright) - 1, 0, wl_all.size - 1)]
d_raw = np.where(np.abs(bright - near) < np.abs(bright - near_lo),
                 bright - near, bright - near_lo)
matched = np.abs(d_raw) < 0.15
d = d_raw[matched]
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(1000 * d, bins=40, alpha=0.55, label='before')
ax.hist(1000 * (d - SHIFT0), bins=40, alpha=0.55, label='after global removal')
ax.axvline(0, color='k', lw=0.8)
ax.axvline(1000 * SHIFT0, color='C0', ls='--', lw=1,
           label=f'global {1000*SHIFT0:+.1f} pm')
ax.set_xlabel('offset to nearest strong db line [pm]'); ax.set_ylabel('bright peaks')
ax.legend(fontsize=8)
ax.set_title('Stage 2: instrument wavelength shift (blind table, global)',
             fontsize=10)
fig.tight_layout()

# raw per-segment medians on the BLIND table — display only; the applied
# per-segment estimate comes from the refined table after stage 3a
segs = np.digitize(bright[matched], SEGMENT_EDGES)
for s in range(3):
    ds = d[segs == s]
    if ds.size >= 3:
        mad = 1000 * np.median(np.abs(ds - np.median(ds)))
        print(f"segment {s} (blind): median {1000 * np.median(ds):+6.1f} pm, "
              f"MAD {mad:.0f} pm ({ds.size} bright peaks)")

global shift = -14.9 pm from 15 anchor lines
segment 0 (blind): median   +0.2 pm, MAD 4 pm (23 bright peaks)
segment 1 (blind): median   -2.5 pm, MAD 8 pm (21 bright peaks)
segment 2 (blind): median  -25.4 pm, MAD 22 pm (16 bright peaks)


## Stage 3a — data-only refinement (`refine_fit(asymmetric='defer')`)

**Consumes** the blind fit + db (shift-corrected). **Produces** the refined table from
pure MODEL EVIDENCE: close same-feature components are tested single vs blend vs
self-absorbed (S/A/B fits + BIC). Blend splits and single-merges act now — they don't
depend on which elements are present. The whole **asymmetric family is deferred**
(`action='deferred'`): claiming self-absorption requires a resonance-capable line of an
element actually in the plasma, and no element posterior exists yet. Stage 3b, after
pass 1, adjudicates them.

Below: every non-trivial decision, then before/after zooms of the biggest ones.

In [5]:
t0 = time.time()
refined, dec_data = refine_fit(x, y, blind, db=db, shift_nm=SHIFT0,
                               asymmetric='defer')
keep('refined', refined)

# per-segment shift from the refined table (significance-gated; falls
# back to the global for segments whose median is within noise of it)
SHIFT, _ = estimate_wavelength_shift_segments(
    refined['sorted_parameter_array'], db)
print(SHIFT)

acted = [d for d in dec_data if d['action'] != 'none']
print(f"{len(acted)} stage-3a decisions (incl. deferred) in {time.time() - t0:.0f}s")
for d in sorted(acted, key=lambda d: d['center']):
    extra = ''
    if 'tau_a' in d:
        extra = (f"  tau={d['tau_a']:.2f}  emission={d['emission_area']:.3g}"
                 f" observed={d['observed_area']:.3g}")
    print(f"  {d['center']:9.3f}  {d['kind']:8s} {d['verdict']:20s} "
          f"{d['action']:8s}{extra}")

diff_peaks('blind', 'refined')

big = sorted([d for d in acted if d['action'] != 'deferred'],
             key=lambda d: -abs(d.get('observed_area', 0) or 0))[:4]
if big:
    fig, axs = plt.subplots(2, 2, figsize=(12, 6.5))
    for ax, d in zip(axs.ravel(), big):
        lo, hi = d['center'] - 0.7, d['center'] + 0.7
        m = (x >= lo) & (x <= hi)
        ax.plot(x[m], (y - bg_of(blind))[m], color='0.15', lw=0.9, label='data - bg')
        ax.plot(x[m], model_of(blind)[m], 'C0-', lw=1.0, label='blind')
        ax.plot(x[m], model_of(refined)[m], 'C3-', lw=1.0, label='refined')
        ax.set_title(f"{d['center']:.2f} nm — {d['verdict']} ({d['action']})", fontsize=9)
        ax.legend(fontsize=7)
    for ax in axs.ravel()[len(big):]:
        ax.axis('off')
    fig.suptitle('Stage 3a: largest data-only refinement actions', fontsize=11)
    fig.tight_layout()

SegmentShift([-18.5(=global), -18.5(=global), -18.5(=global)] pm, global -18.5 pm, n=(4, 8, 10))
40 stage-3a decisions (incl. deferred) in 12s
    198.789  pair     asymmetric           deferred  tau=3.41  emission=183 observed=44.4
    221.713  pair     asymmetric-displaced deferred  tau=2.57  emission=300 observed=70.5
    240.456  residual asymmetric-displaced deferred  tau=3.11  emission=462 observed=80.2
    251.692  pair     blend                split   
    254.237  pair     asymmetric           deferred  tau=3.57  emission=198 observed=28.3
    254.779  pair     asymmetric-nonresonant deferred  tau=5.34  emission=454 observed=39.6
    261.203  pair     asymmetric           deferred  tau=3.30  emission=465 observed=78.5
    271.935  pair     asymmetric           deferred  tau=3.12  emission=206 observed=39.1
    278.193  pair     single               merge   
    279.578  pair     asymmetric-displaced deferred  tau=2.05  emission=2.55e+03 observed=852
    280.239  pair     asymm

## Stage 4 — pass-1 indexer (`PeakyIndexerV3.run`)

**Consumes** refined peaks in the db frame (+ per-shot nₑ prior
from the H-α Stark width when present). **Produces** the first physical solve: plasma
(T, nₑ), per-(element, ion) concentrations under Saha–Boltzmann coupling with
response-side self-absorption anchoring on resonance doublets (K I 766/770, Na D, …),
and the pass-1 **established elements** (fraction ≥ 0.002) that license minor-line seeding.

The scatter shows, per fitted peak, observed vs. predicted amplitude colored by the
dominant species in the solve — tight diagonal = whole-pattern consistency; the
composition is only as good as this agreement.

Pass 1 is deliberately treated as *provisional*: it runs on the
sparsest peak table (no seeded/recovered lines yet), so its basin can be wrong —
in this very run it established only C/Bi at low T. Its only job is to license
minor-line seeding; pass 2, on the enriched table, is the identification that counts.


In [6]:
rpeaks = refined['sorted_parameter_array']
NE_INIT, NE_BOUNDS = _halpha_ne(rpeaks)
print(f"H-alpha ne prior: {NE_INIT if NE_INIT else 'absent'}"
      + (f"  bounds={NE_BOUNDS}" if NE_INIT else ''))


def db_frame(p):
    out = p.copy()
    out[:, 1] -= shift_at(SHIFT, out[:, 1])
    return out


idx_kwargs = dict(dbpath=DB_PATH)
run_kwargs = dict(sa_doublets=True, n_calls=N_CALLS, verbose=False,
                  sa_stimulated_emission=False)
if NE_INIT is not None:
    idx_kwargs['ne_init'] = NE_INIT
    run_kwargs['ne_bounds'] = NE_BOUNDS

t0 = time.time()
idx1 = PeakyIndexerV3(db_frame(rpeaks), **idx_kwargs)
res1 = idx1.run(**run_kwargs)
RESULTS['pass 1'] = res1
COMPOSITIONS.append(('pass 1', dict(res1.element_fractions)))
established = sorted([el for el, f in res1.element_fractions.items()
                      if f >= ESTABLISHED_MIN_FRACTION], key=element_sort_key)
print(f"pass 1: T={res1.temperature:.0f} K  log ne={res1.ne:.2f}  "
      f"r2={res1.r_squared:.3f}  ({time.time() - t0:.0f}s)")
print(f"established: {established}")

comp_chart([('pass 1', res1.element_fractions)], 'Stage 4: pass-1 composition')

# observed vs predicted amplitude, colored by dominant species
A = np.asarray(idx1._last_A)[:rpeaks.shape[0]]
contrib = A * res1.concentrations
dom = np.argmax(contrib, axis=1)
obs, pred = res1.observed, res1.predicted
fig, ax = plt.subplots(figsize=(6.4, 5.6))
seen = {}
for j in range(len(obs)):
    el = res1.species[dom[j]].element if contrib[j].max() > 0 else '?'
    c = element_color(el) if el != '?' else '0.6'
    lbl = el if el not in seen else None
    seen[el] = True
    ax.loglog(max(obs[j], 1), max(pred[j], 1), 'o', ms=4, color=c,
              mec='0.2', mew=0.3, label=lbl)
lim = [max(min(obs.min(), 1), 1), obs.max() * 2]
ax.plot(lim, lim, 'k--', lw=0.8)
ax.set_xlabel('observed amplitude'); ax.set_ylabel('predicted amplitude')
ax.legend(fontsize=7, ncol=3, title='dominant species', title_fontsize=8)
ax.set_title(f'Stage 4: whole-pattern agreement (r2={res1.r_squared:.3f})', fontsize=10)
fig.tight_layout()

H-alpha ne prior: 17.23682530475796  bounds=(16.93682530475796, 17.53682530475796)


pass 1: T=6027 K  log ne=16.94  r2=0.819  (21s)
established: ['C', 'Si', 'Ti', 'Bi']


## Stage 3b — physics adjudication of the deferred asymmetric features

**Consumes** the 3a table + the pass-1 posterior. The posterior used is the
candidate-species set the whole-pattern solve *retained* (broader than the established
list — a wrong pass-1 basin must not veto a real resonance line's merge; still far
tighter than "any line in the periodic table", which is what the old single-pass
refinement had to gate against). Each deferred feature is re-classified with the
resonance gates conditioned on those elements; surviving `asymmetric` verdicts merge
now. The merges' zones and their measured emission/observed ratios feed every later
stage: seeding/recovery/deblending exclude the zones, and stage 10b propagates the
ratios for species the doublet channel does not anchor.

In [7]:
posterior = sorted({sp.element for sp in res1.species})
print(f"posterior ({len(posterior)} candidate elements):", ' '.join(posterior))
t0 = time.time()
refined, dec_phys = refine_fit(x, y, refined, db=db,
                               elements=posterior or None,
                               shift_nm=SHIFT, asymmetric='only')
keep('adjudicated', refined)
decisions = dec_data + dec_phys
merges = [d for d in dec_phys if d['action'] == 'merge']
print(f"{len(merges)} asymmetric merges adjudicated in {time.time() - t0:.0f}s")
for d in sorted(merges, key=lambda d: d['center']):
    print(f"  {d['center']:9.3f}  tau={d['tau_a']:5.2f}  "
          f"observed={d['observed_area']:8.1f}  emission={d['emission_area']:8.1f}"
          f"  (x{d['emission_area']/max(d['observed_area'],1e-12):.2f})")

# merge zones + pre-measured emission ratios for the downstream stages
sa_zones, sa_merges = [], []
for dec in decisions:
    if (dec.get('action') == 'merge'
            and str(dec.get('verdict', '')).startswith('asymmetric')
            and dec.get('params_asym') is not None):
        pA = dec['params_asym']
        halfw = 1.5 * max(float(voigt_width(max(pA[2], 1e-6),
                                            max(pA[3], 1e-6))), 0.15)
        sa_zones.append((float(pA[1]), halfw))
        obs_a = float(dec.get('observed_area') or 0.0)
        if obs_a > 0.0 and dec.get('emission_area'):
            sa_merges.append(dict(center_nm=float(pA[1]),
                                  factor=float(dec['emission_area']) / obs_a,
                                  tau_a=float(dec.get('tau_a', 0.0)),
                                  observed_area=obs_a,
                                  emission_area=float(dec['emission_area'])))
print(f"{len(sa_zones)} exclusion zones, {len(sa_merges)} pre-measured ratios")

for d in sorted(merges, key=lambda d: -d['observed_area'])[:2]:
    overlay(d['center'], 1.2, stages=('refined', 'adjudicated'))
    plt.gca().set_title(f"3b merge at {d['center']:.2f} nm  tau={d['tau_a']:.2f}",
                        fontsize=9)

posterior (17 candidate elements): Al Ba Be Bi C Ca Cs Fe K Li Mg Mn Na Rb Si Sr Ti


8 asymmetric merges adjudicated in 7s
    254.237  tau= 3.58  observed=    28.1  emission=   197.8  (x7.04)
    261.203  tau= 3.30  observed=    78.5  emission=   464.5  (x5.92)
    280.239  tau= 0.58  observed=   562.3  emission=   757.9  (x1.35)
    394.400  tau= 0.82  observed=   401.1  emission=   651.5  (x1.62)
    407.749  tau= 0.49  observed=  1367.7  emission=  1833.8  (x1.34)
    670.746  tau= 0.58  observed=  1492.3  emission=  2099.1  (x1.41)
    766.496  tau= 0.46  observed=  1096.6  emission=  1380.9  (x1.26)
    769.868  tau= 0.38  observed=   684.8  emission=   850.2  (x1.24)
8 exclusion zones, 8 pre-measured ratios


## Stage 5 — Boltzmann-prior minor lines (`seed_minor_lines`)

**Consumes** the adjudicated fit + the pass-1 established-element list + the merge
exclusion zones (a seed inside a merge zone would convert its deliberate core residual
into a phantom satellite and erode the merged row's observed area — measured 21–93%
loss before the gate existed).
**Produces** new small components at database positions of those elements' still-unfitted
lines, where each element's own bright lines set the expected intensity scale (a ≥2σ bump
must exist at the predicted position; per-line Boltzmann-SNR and BIC gates). This is
*prior-driven*: only established elements may seed, so noise cannot vote elements in.

When pass 1 lands in a spurious basin (as above), few or no lines seed here —
the element-agnostic residual recovery (stage 6) then carries the enrichment, and the
corroboration pass (stage 9) re-seeds from the trustworthy pass-2 element set.


In [8]:
t0 = time.time()
seeded, seed_records = seed_minor_lines(x, y, refined, db, established,
                                        shift_nm=SHIFT,
                                        exclude=tuple(sa_zones))
keep('seeded', seeded)
added = [r for r in seed_records if r['action'] == 'added']
print(f"{len(added)} minor lines added in {time.time() - t0:.0f}s")
per_el = {}
for r in added:
    per_el.setdefault(r['element'], []).append(r)
for el in sorted(per_el, key=element_sort_key):
    rows = per_el[el]
    print(f"  {el:2s}: {len(rows):2d} lines  "
          + ', '.join(f"{r['wavelength_db']:.1f}" for r in rows[:8])
          + (' ...' if len(rows) > 8 else ''))

if added:
    fig, ax = plt.subplots(figsize=(11, 2.8))
    ax.plot(x, y - bg_of(seeded), color='0.8', lw=0.4)
    for r in added:
        w = r['wavelength_db'] + float(shift_at(SHIFT, r['wavelength_db']))
        ax.axvline(w, color=element_color(r['element']), lw=1.0, alpha=0.8)
    ax.set_yscale('log'); ax.set_ylim(bottom=1)
    ax.set_xlabel('wavelength [nm]')
    ax.set_title(f'Stage 5: {len(added)} seeded minor lines (colored by element)',
                 fontsize=10)
    fig.tight_layout()

    for r in sorted(added, key=lambda r: -r['snr'])[:3]:
        w = r['wavelength_db'] + float(shift_at(SHIFT, r['wavelength_db']))
        overlay(w, 1.2, stages=('adjudicated', 'seeded'))
        plt.gca().set_title(f"seeded {r['element']} at {r['wavelength_db']:.2f} nm "
                            f"(snr {r['snr']:.1f})", fontsize=9)

6 minor lines added in 2s
  Si:  1 lines  198.3
  Ti:  5 lines  332.9, 316.9, 319.1, 322.3, 333.5


## Stage 6 — element-agnostic residual recovery (`recover_residual_lines`)

**Consumes** the seeded fit + the merge zones (excluded — their core residual is
deliberate). **Produces** components fitted at significant (>4σ) positive residual
maxima *from the data alone* — real lines the seeder could not predict, most often a
line-rich element (Fe) whose per-stage Boltzmann scale failed the trust gate. The
pass-2 indexer identifies them afterwards.

In [9]:
resid_before = (y - bg_of(seeded) - model_of(seeded)) / np.maximum(noise_of(seeded), 1e-12)
t0 = time.time()
recovered_fit, recovered = recover_residual_lines(x, y, seeded,
                                                  exclude=tuple(sa_zones))
keep('recovered', recovered_fit)
rec_added = [r for r in recovered if r['action'] == 'added']
rej = [r for r in recovered if r['action'] == 'rejected']
print(f"{len(rec_added)} residual lines recovered, {len(rej)} rejected "
      f"({time.time() - t0:.0f}s)")
resid_after = (y - bg_of(recovered_fit) - model_of(recovered_fit))     / np.maximum(noise_of(recovered_fit), 1e-12)
print(f"pixels above +2 sigma: {int((resid_before > 2).sum())} -> "
      f"{int((resid_after > 2).sum())}")

for r in sorted(rec_added, key=lambda r: -r['delta_bic'])[:3]:
    overlay(r['center'], 1.2, stages=('seeded', 'recovered'))
    plt.gca().set_title(f"recovered {r['center']:.2f} nm  snr={r['snr']:.1f}  "
                        f"dBIC={r['delta_bic']:.0f}", fontsize=9)
if rej:
    print('still-unexplained residual maxima (rejected):')
    for r in sorted(rej, key=lambda r: -r.get('resid0', 0))[:8]:
        print(f"  {r['center0']:9.3f}  resid={r['resid0']:7.0f}  snr0={r['snr0']:5.1f}")

13 residual lines recovered, 0 rejected (2s)
pixels above +2 sigma: 658 -> 610


## Stage 7 — shape QC + shoulder deblends (`alibz.profiles`, `deblend_shoulders`)

**Consumes** the current fit. **Produces** (a) a per-peak shape
classification against each detector segment's instrument width floor —
`instrumental` / `broadened` / `narrow` / `shoulder` / `sa-like` / `irregular` — and
(b) two-component refits of `shoulder`-flagged peaks (a one-sided flank bump is an
unresolved overlapping line contaminating the fitted area). Deblending runs **before**
identification so passes 2–3 see decontaminated areas. `sa-like` peaks are left alone
here — they are handled on the response side (doublet anchors) or in stage 10.

In [10]:
prof_pre = analyze_peak_profiles(x, y, recovered_fit)
print('shape classes:', profile_summary(prof_pre))

cls_colors = {'instrumental': 'C2', 'broadened': 'C0', 'narrow': 'C4',
              'shoulder': 'C1', 'sa-like': 'C3', 'irregular': '0.5',
              'unresolved-window': '0.8'}
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for cls, c in cls_colors.items():
    rows = [r for r in prof_pre if r['classification'] == cls]
    if not rows:
        continue
    ax.scatter([r['width_ratio'] for r in rows],
               [abs(r['core_defect_sigma'] or 0) + 0.05 for r in rows],
               s=14, color=c, label=f"{cls} ({len(rows)})", alpha=0.75)
ax.axvline(1.0, color='0.6', lw=0.7, ls='--')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('width ratio (FWHM / segment instrument floor)')
ax.set_ylabel(r'|core defect| [$\sigma$] (+0.05)')
ax.legend(fontsize=8)
ax.set_title('Stage 7: peak-shape physics per fitted component', fontsize=10)
fig.tight_layout()

t0 = time.time()
deblended_fit, deb_records = deblend_shoulders(x, y, recovered_fit, prof_pre,
                                               exclude=tuple(sa_zones))
keep('deblended', deblended_fit)
deb = [r for r in deb_records if r['action'] == 'deblended']
print(f"{len(deb)} shoulders deblended in {time.time() - t0:.0f}s")
for r in deb:
    print(f"  {r['center_nm']:9.3f} -> new component at {r['new_center_nm']:.3f}  "
          f"area={r['area_new']:.1f}  snr={r['snr']:.1f}  dBIC={r['delta_bic']:.0f}")
for r in deb[:3]:
    overlay(r['center_nm'], 1.0, stages=('recovered', 'deblended'))
    plt.gca().set_title(f"deblended shoulder at {r['center_nm']:.2f} nm", fontsize=9)
    components('deblended', r['center_nm'], 1.0)

shape classes: {'instrumental': 250, 'sa-like': 23, 'shoulder': 9, 'irregular': 12, 'broadened': 15, 'narrow': 5}
4 shoulders deblended in 0s
    407.233 -> new component at 407.817  area=98.1  snr=12.1  dBIC=11
    588.667 -> new component at 589.783  area=63.3  snr=19.9  dBIC=143
    517.268 -> new component at 517.450  area=10.7  snr=8.7  dBIC=15
    279.751 -> new component at 280.000  area=36.1  snr=12.5  dBIC=90


/var/folders/yj/sjr_202x2fvfp9wq2lrrdf0h0000gn/T/ipykernel_71867/2590290261.py:75: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(10, 3.6))


## Stage 8 — pass-2 indexer (confirm elements)

**Consumes** the enriched, deblended peak table (db frame),
warm-started at the pass-1 plasma state. **Produces** the *confirmed* element set and an
updated plasma solve — pass 1 decided who may seed; pass 2 decides who is actually present
now that the seeded/recovered/deblended evidence is in the table.

In [11]:
fpeaks = deblended_fit['sorted_parameter_array']
t0 = time.time()
idx2 = PeakyIndexerV3(db_frame(fpeaks), dbpath=DB_PATH,
                      temperature_init=res1.temperature, ne_init=res1.ne)
res2 = idx2.run(**run_kwargs)
RESULTS['pass 2'] = res2
COMPOSITIONS.append(('pass 2', dict(res2.element_fractions)))
print(f"pass 2: T={res2.temperature:.0f} K  log ne={res2.ne:.2f}  "
      f"r2={res2.r_squared:.3f}  ({time.time() - t0:.0f}s)")
print(f"dT={res2.temperature - res1.temperature:+.0f} K  "
      f"dlog ne={res2.ne - res1.ne:+.2f} vs pass 1")
comp_chart([('pass 1', res1.element_fractions),
            ('pass 2', res2.element_fractions)],
           'Stage 8: composition, pass 1 vs pass 2')

pass 2: T=5844 K  log ne=17.54  r2=0.786  (18s)
dT=-183 K  dlog ne=+0.60 vs pass 1


<Figure size 924x380 with 1 Axes>

## Stages 9–10 — iterative deepening (seed + guarded recover + fixed re-solve)

**Consumes** the pass-2 confirmed elements. The ions **quantified from their intense
peaks** — those with ≥ `CONFIDENT_MIN_REFS` clean reference lines in some stage (Fe here
has 60/46, Si 16, Ti 17/20) — have earned a *lower* recovery bar on their own faint lines.
Each round, at a progressively lower local-noise bar (`DEEPEN_BARS` = 3 → 2.5 → 2σ):

- **seeds** those ions' still-unfitted lines (Boltzmann prior, robust scale), and
- **recovers** faint residuals *element-agnostically but only near a confident ion's own
  database line* (`supported_lines` → the coverage map). A bump sitting on an anchored
  ion's line is very likely that ion's faint line, not noise — so its bar drops to 2σ,
  while empty regions keep the full 4σ and cannot admit chance coincidences.

The composition is then **re-solved at the FIXED pass-2 plasma state**
(`PeakyIndexerV3.solve_at` — no `gp_minimize`, so it cannot drift into a different
(T, nₑ) basin; the weak lines only corroborate what the intense lines already fixed). A
round is rejected wholesale if it *newly* collapses onto one element
(`composition_collapsed`, guarding against the measured K 0.54 → Hg 1.00 / Si 0.35 → Fe
0.99 failures). This replaces the old single pass-3 `gp_minimize` re-index, which *was*
the basin-drift risk the guard existed to catch.

In [12]:
confirmed = sorted([el for el, f in res2.element_fractions.items()
                    if f >= ESTABLISHED_MIN_FRACTION], key=element_sort_key)
final_fit, fidx, result = deblended_fit, idx2, res2

# confident ions: quantified from intense peaks (>= CONFIDENT_MIN_REFS refs)
scales, _ = match_and_scale(deblended_fit['sorted_parameter_array'], db,
                            confirmed, shift_nm=SHIFT)
confident = sorted({el for (el, _s), info in scales.items()
                    if info['n_ref'] >= CONFIDENT_MIN_REFS}, key=element_sort_key)
print(f"confirmed: {confirmed}")
print(f"confident (>= {CONFIDENT_MIN_REFS} clean refs): {confident}")

# coverage map: instrument-frame db lines of the confident ions
sup = []
for el in confident:
    if el in db.no_lines:
        continue
    arr = db.lines(el)
    if arr.size == 0:
        continue
    mk = (arr[:, 0].astype(float) <= 2) & (arr[:, 3].astype(float) >= SUPPORT_GA_FLOOR)
    wl = arr[mk, 1].astype(float)
    if wl.size:
        sup.append(wl + shift_at(SHIFT, wl))
supported = np.concatenate(sup) if sup else np.empty(0)

work = deblended_fit
for bar in DEEPEN_BARS:
    if not confident:
        break
    t0 = time.time()
    work, corr = seed_minor_lines(x, y, work, db, confident, shift_nm=SHIFT,
                                  accept_snr=bar, min_expected_snr=bar,
                                  robust_elements=set(confident),
                                  exclude=tuple(sa_zones))
    n_seed = sum(1 for r in corr if r['action'] == 'added')
    work, rec = recover_residual_lines(x, y, work, exclude=tuple(sa_zones),
                                       supported_lines=supported,
                                       snr_min_supported=bar, accept_snr_supported=bar,
                                       support_tol_nm=SUPPORT_TOL_NM)
    n_rec = sum(1 for r in rec if r['action'] == 'added')
    if n_seed + n_rec == 0:
        print(f"bar {bar}: nothing added")
        continue
    idxN = PeakyIndexerV3(db_frame(work['sorted_parameter_array']), dbpath=DB_PATH,
                          temperature_init=res2.temperature, ne_init=res2.ne)
    idxN.build_candidate_matrix(sa_doublets=True)
    resN = idxN.solve_at(res2.temperature, res2.ne, res2.sigma, res2.gamma)
    if composition_collapsed(res2.element_fractions, resN.element_fractions):
        print(f"bar {bar}: +{n_seed} seed +{n_rec} recover -> COLLAPSE, stop")
        break
    final_fit, fidx, result = work, idxN, resN
    COMPOSITIONS.append((f'deepen {bar}', dict(resN.element_fractions)))
    print(f"bar {bar}: +{n_seed} seed +{n_rec} recover  top->"
          f"{max(resN.element_fractions.values()):.3f}  ({time.time()-t0:.0f}s)")
keep('deepened', final_fit)
STAGES['final'] = final_fit
comp_chart([('pass 2', res2.element_fractions),
            ('deepened', result.element_fractions)],
           'Stages 9-10: composition after iterative deepening')
print(f"deepened: T={result.temperature:.0f} K (fixed at pass-2)  "
      f"r2={result.r_squared:.3f}  peaks {res2.observed.shape[0]}->"
      f"{result.observed.shape[0]}")

confirmed: ['Li', 'B', 'Mg', 'Al', 'Si', 'K', 'Ti', 'Fe', 'Eu', 'Hg']
confident (>= 4 clean refs): ['B', 'Mg', 'Al', 'Si', 'K', 'Ti', 'Fe', 'Eu', 'Hg']


bar 3.0: +21 seed +5 recover  top->0.472  (54s)


bar 2.5: +2 seed +9 recover  top->0.467  (61s)


bar 2.0: +0 seed +4 recover  top->0.456  (70s)
deepened: T=5844 K (fixed at pass-2)  r2=0.636  peaks 318->359


## Stage 10b — SA growth-curve area recovery (`recover_sa_areas`)

**Consumes** the final fit's shape records + the accepted plasma
state. For each `sa-like` peak whose dominant species is **not** already anchored through
the indexer's resonance-doublet channel (K I 766/770, Na D, … — correcting those again
would double-count), the peak is refit with the self-absorption model (`sa_voigt`, whose
area parameter IS the unattenuated emission area) against a symmetric control. Acceptance:
ΔBIC ≥ 10, τ below ceiling, amplification ≤ 5×.

Accepted factors correct the observed amplitudes and the composition is **re-solved
linearly at the fitted (T, nₑ)** — no new Bayesian pass, hence no basin risk; a corrected
composition that newly collapses is rejected wholesale.

The stage-3b merges' **pre-measured emission/observed ratios** join the same re-solve
(`premeasured=`): a merged row carries the observed (attenuated) area, its zone is
excluded from the growth-curve refit, and its species — when not doublet-anchored —
would otherwise never receive the correction the asymmetric fit already measured
(audited: Li I ×1.4, Mg I ×1.5, Al I ×1.6 dropped per MW2-112 spectrum before this
channel existed).

In [13]:
from alibz.refinement import sa_voigt
profiles = analyze_peak_profiles(x, y, final_fit)
print('final shape classes:', profile_summary(profiles))
pre_sa_fractions = dict(result.element_fractions)
result, sa_records, sa_used = recover_sa_areas(fidx, result, x, y, final_fit,
                                               profiles, exclude=tuple(sa_zones),
                                               premeasured=tuple(sa_merges))
by_action = {}
for r in sa_records:
    by_action.setdefault(r['action'], []).append(r)
print({k: len(v) for k, v in by_action.items()},
      f"-> correction {'APPLIED' if sa_used else 'not applied'}")

for r in by_action.get('sa-recovered', []):
    src = r.get('source', 'growth-curve')
    print(f"  {r['center_nm']:9.3f} [{r.get('species', '?'):>6s}] "
          f"observed={r['observed_area']:.1f} -> emission={r['emission_area']:.1f} "
          f"(x{r['factor']:.2f}, tau={r['tau_a']:.2f}, {src})")
if by_action.get('anchored'):
    print('  anchored (left to the indexer doublet channel): '
          + ', '.join(f"{r['center_nm']:.1f} {r.get('species', '')}"
                      for r in by_action['anchored']))

pk_f = np.atleast_2d(final_fit['sorted_parameter_array'])
for r in by_action.get('sa-recovered', [])[:3]:
    j = int(r['index'])
    mu, s0, g0 = pk_f[j, 1], max(pk_f[j, 2], 1e-6), max(pk_f[j, 3], 1e-6)
    lo, hi = mu - 0.8, mu + 0.8
    m = (x >= lo) & (x <= hi)
    fig, ax = plt.subplots(figsize=(8, 3.4))
    ax.plot(x[m], (y - bg_of(final_fit))[m], color='0.15', lw=1.0, label='data - bg')
    ax.plot(x[m], multi_voigt(x[m], pk_f[j, :4]), 'C0-', lw=1.1,
            label=f"symmetric Voigt (observed area {r['observed_area']:.0f})")
    ax.plot(x[m], sa_voigt(x[m], r['emission_area'], mu, s0, g0,
                           r['tau_a'], r.get('delta_nm', 0.0)), 'C3--', lw=1.1,
            label=f"SA model (emission area {r['emission_area']:.0f}, "
                  f"tau={r['tau_a']:.2f})")
    ax.plot(x[m], multi_voigt(x[m], np.array([r['emission_area'], mu, s0, g0])),
            'C3:', lw=0.9, alpha=0.7, label='unattenuated emission (reconstructed)')
    ax.legend(fontsize=7.5)
    ax.set_title(f"Stage 10b: SA recovery at {mu:.2f} nm  (x{r['factor']:.2f})",
                 fontsize=10)
    ax.set_xlabel('wavelength [nm]')
    fig.tight_layout()

if sa_used:
    COMPOSITIONS.append(('SA re-solved', dict(result.element_fractions)))
    comp_chart([('before SA recovery', pre_sa_fractions),
                ('after linear re-solve', result.element_fractions)],
               'Stage 10b: composition after SA area recovery')

final shape classes: {'sa-like': 18, 'shoulder': 9, 'instrumental': 294, 'irregular': 16, 'broadened': 18, 'narrow': 4}


{'excluded': 1, 'anchored': 19, 'rejected': 3, 'sa-recovered': 3} -> correction APPLIED
    610.293 [  Li I] observed=99.2 -> emission=130.2 (x1.31, tau=0.51, growth-curve)
    767.006 [     ?] observed=28.4 -> emission=141.8 (x5.00, tau=1.67, growth-curve)
    670.723 [  Li I] observed=1492.3 -> emission=2099.1 (x1.41, tau=0.58, refinement-merge)
  anchored (left to the indexer doublet channel): 393.4 Ca II, 766.5 K I, 769.9 K I, 280.2 Fe I, 393.3 Ca II, 394.4 Al I, 259.9 Fe I, 407.8 Sr II, 383.6 Fe I, 280.7 Fe I, 254.2 Ti I, 392.8 Fe I, 766.5 K I, 280.2 Fe I, 261.2 Ti I, 254.2 Ti I, 407.7 Sr II, 769.9 K I, 394.4 Al I


## Stage 11 — detections, confounders, shape flags (`analyze_detections`)

**Consumes** the final indexer + result, per-peak area
uncertainties (joint-GLS blend groups), and — when SA recovery was applied — the SAME
corrected amplitudes the re-solved composition came from. **Produces** the per-element
detection report: status (detected / single-line / blended-only / confounded / marginal /
weak / upper-limit), z-scores from amplitude resampling at fixed (T, nₑ), the
true-negative confounder resolution (`fraction_resolved`), and per-element shape QC
(`sa_share`, `shoulder_share`, `clean_anchors`).

In [14]:
bg = bg_of(final_fit)
fpeaks_f = final_fit['sorted_parameter_array']
area_sigma = estimate_peak_uncertainties(x, y - bg, fpeaks_f)[:, 0]
amp_stash = None
if sa_used:
    amp_stash = fidx._obs_amp.copy()
    for r in sa_records:
        if r['action'] == 'sa-recovered':
            fidx._obs_amp[int(r['index'])] *= float(r['factor'])
try:
    t0 = time.time()
    det = analyze_detections(fidx, result, area_sigma, shift=SHIFT,
                             dbpath=DB_PATH, draws=16)
finally:
    if amp_stash is not None:
        fidx._obs_amp = amp_stash
print(f"detections in {time.time() - t0:.0f}s")
shape_quality = element_shape_quality(det.get('support_idx', {}), profiles)
COMPOSITIONS.append(('confounder-resolved', {
    el: f for el, f in det['resolved_fractions'].items() if f > 0}))

print(f"{'el':>4} {'status':>12} {'fraction':>9} {'resolved':>9} {'z':>6} "
      f"{'lines':>5} {'confounder':>10} {'sa_share':>8} {'clean':>5}")
for d in sorted(det['detections'], key=lambda d: -(d['fraction'] or 0)):
    q = shape_quality.get(d['element'], {})
    print(f"{d['element']:>4} {d['status']:>12} {d['fraction']:9.5f} "
          f"{d.get('fraction_resolved', d['fraction']):9.5f} {d['z']:6.1f} "
          f"{d['n_lines']:5d} {d.get('confounder') or '':>10} "
          f"{q.get('sa_share', 0):8.2f} {q.get('clean_anchors', ''):>5}")

comp_chart([('as fit (raw NNLS)', result.element_fractions),
            ('confounder-resolved', det['resolved_fractions'])],
           'Stage 11: raw vs true-negative-resolved composition')

detections in 8s
  el       status  fraction  resolved      z lines confounder sa_share clean
  Si     detected   0.45531   0.45531    8.2    17         Bi     0.00    16
  Hg     marginal   0.27324   0.27324    2.5     1                1.00     0
  Al     marginal   0.09083   0.09083    3.0    11         Mn     0.26    10
  Fe     detected   0.05155   0.05453    6.7    54         Eu     0.00    54
  Mn         weak   0.02694   0.02694    0.6     3         Mg     0.00     1
  Ti         weak   0.02253   0.02253    1.5    23         Mn     0.00    22
  Eu         weak   0.02229   0.02229    0.4     0                0.00      
   B         weak   0.02219   0.02219    1.2     1                0.00     1
   K     detected   0.01136   0.01136    7.0     2                1.00     0
  Li     detected   0.00855   0.00855    7.8     3                0.99     1
  Mg  single-line   0.00614   0.00614    6.9     1                0.00     0
  Be   confounded   0.00298   0.00000    6.8     1         

<Figure size 924x380 with 1 Axes>

## Composition evolution across the whole chain

Every element's abundance at each quantification checkpoint (pass 1, pass 2, each adopted
`deepen <bar>` round, then the SA re-solve and the confounder resolution). Watch for:
elements that appear only after seeding/recovery, the drift across the deepening rounds
(each is a fixed-(T,nₑ) re-solve, so moves come from the added lines, not a moved plasma
state), and confounded elements collapsing into their rival at the resolution step. A
sudden single-element takeover between checkpoints is the basin-collapse signature the
guard exists for.

In [15]:
els = sorted({el for _, fr in COMPOSITIONS for el, f in fr.items() if f >= 3e-3},
             key=element_sort_key)
fig, ax = plt.subplots(figsize=(9.5, 5.2))
xi = np.arange(len(COMPOSITIONS))
for el in els:
    vals = [max(fr.get(el, 0.0), 1e-5) for _, fr in COMPOSITIONS]
    ax.plot(xi, vals, 'o-', color=element_color(el), lw=1.4, ms=5, label=el)
    ax.annotate(f' {el}', (xi[-1], vals[-1]), fontsize=8,
                color=element_color(el), va='center')
ax.set_yscale('log')
ax.set_xticks(xi)
ax.set_xticklabels([lab for lab, _ in COMPOSITIONS], rotation=20, fontsize=9)
ax.set_ylabel('atom fraction')
ax.set_ylim(bottom=1e-4)
ax.set_title('Element fractions at each quantification checkpoint', fontsize=11)
ax.grid(alpha=0.25, axis='y')
fig.tight_layout()

print(f"final: T={result.temperature:.0f} K  log ne={result.ne:.2f}  "
      f"r2={result.r_squared:.3f}")
print('flags: stage_disagreement > 0.5:',
      {el: round(v, 2) for el, v in result.stage_disagreement.items()
       if np.isfinite(v) and v > 0.5 and result.element_fractions.get(el, 0) > 0})

final: T=5844 K  log ne=17.54  r2=0.689
flags: stage_disagreement > 0.5: {'Ca': 1.0, 'Sr': 0.99, 'Ba': 0.92}


## Interactive exploration

`explore(center, span)` stacks everything this notebook knows about a window: all captured
stage models over the data, the final fit's individual components, database-line positions
of the detected elements (rug at the top), and the windowed peak table. Two worked
examples below — then the widget cell for free browsing.

In [16]:
# the Mg II 279.6/280.3 resonance region: refinement, seeding, deblending and the
# Mg/Mn confounder story all play out here
explore(280.0, 2.6)

final: peaks in [278.70, 281.30] nm
   center       area   sigma   gamma  FWHM_pm
  280.222      562.3  0.0404  0.0395    144.3
  279.500      540.0  0.0863  0.0000    203.2
  279.751      243.3  0.0835  0.0007    197.4
  279.043      145.8  0.0806  0.0129    203.9
  280.713       55.4  0.0010  0.1004    200.9
  279.264       39.5  0.0570  0.0000    134.2
  280.417       38.7  0.0819  0.0000    192.7
  280.000       36.1  0.0629  0.0007    148.8
  278.745       33.2  0.0673  0.0000    158.4


In [17]:
# the dense Fe I/II forest at 248.3-249.5 nm: two resonance-capable Fe I
# lines 250 pm apart that near-degenerate statistics once merged into a
# fictitious self-absorbed line (fixed: db-supported blends can no longer
# be merged as SA, and exclude-zone rows are frozen in neighbouring
# refits). The small ~248.55 nm bump below recovery's 4-sigma prominence
# bar is a known unmodeled remainder.
explore(248.9, 2.4)

final: peaks in [247.70, 250.10] nm
   center       area   sigma   gamma  FWHM_pm
  249.046      117.0  0.0010  0.0902    180.4
  248.298       71.5  0.0637  0.0000    150.0
  248.757       59.1  0.0441  0.0536    172.5
  247.867       41.3  0.0000  0.1000    199.9
  249.756       25.9  0.0924  0.0010    218.7
  249.309       18.4  0.0570  0.0000    134.2
  250.096       12.9  0.0608  0.0000    143.3


In [18]:
try:
    import ipywidgets as W

    def _view(center, span, stages, logscale):
        overlay(center, span, stages=list(stages) or None, log=logscale)
        plt.show()

    W.interact(_view,
               center=W.FloatSlider(min=float(x[0]), max=float(x[-1]), step=0.05,
                                    value=280.0, description='center [nm]',
                                    readout_format='.2f',
                                    layout=W.Layout(width='70%')),
               span=W.FloatLogSlider(base=10, min=-0.3, max=2.9, value=2.5,
                                     description='span [nm]',
                                     layout=W.Layout(width='70%')),
               stages=W.SelectMultiple(options=list(STAGES),
                                       value=('blind', 'final'),
                                       description='stages'),
               logscale=W.Checkbox(value=False, description='log y'))
except ImportError:
    print('ipywidgets not installed - `pip install ipywidgets` for sliders.\n'
          'Interactive fallback: call explore(center_nm, span_nm), '
          'overlay(...), components(...), diff_peaks(...) from any cell.')

interactive(children=(FloatSlider(value=280.0, description='center [nm]', layout=Layout(width='70%'), max=960.…

## Debugging map: which stage to interrogate for which pathology

| symptom | stage to inspect | what to look at |
|---|---|---|
| broad wings unfit / background swallowed a line | 1 (blind fit) | overview residual panel, background curve |
| element matched to wrong lines | 2 (shift) | offset histogram width/bimodality; gated per-segment shifts (`SHIFT`) |
| one fat asymmetric line quantified as two elements | 3a/3b (refinement) | 3a model evidence (deferred verdicts); 3b posterior-conditioned merge/veto |
| self-absorbed line's abundance too low | 3b/10b | was its merge vetoed (element missing from posterior)? was its emission ratio propagated or `anchored`? |
| composition dominated by one element, r² fine | 4/8/9 (indexer passes) | evolution chart for a sudden takeover; basin-guard verdict; `dominant-weak-shape` logic in stage 11 |
| known trace element missing | 5–6 / 9 (seeding/recovery/deepening) | established in pass 1? confident by pass 2? seeded/recovered lines rejected? |
| faint line of a confident ion unmodeled | 9 (deepening) | is the ion in `confident`? is its db line in the coverage map (gA ≥ `SUPPORT_GA_FLOOR`)? is there a ≥2σ residual there at all? |
| abundance rests on a distorted line | 7/10b (profiles/SA) | classification of its support peaks, `sa_share`, recovery factors |
| two elements swap depending on sample | 11 (confounders) | `confounded` status, contested share, resolved fraction |

The production entry point is `alibz.pipeline.analyze_spectrum` — this notebook mirrors it
stage-for-stage (same calls, same kwargs); if the two ever disagree, the pipeline is the
source of truth and this notebook needs updating.